Basic Imports

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Load the dataset

In [ ]:
data = pd.read_csv("../data/raw/insurance.csv")




data.head()

In [ ]:
data.shape

In [ ]:
data.info()

Label Encode Object Types

In [ ]:
d_types = dict(data.dtypes)
for name , type_ in d_types.items():
    if str(type_) == 'object':
        print(f"<======== {name} ===========>")
        print(data[name].value_counts())
        print()

In [ ]:
from sklearn.preprocessing import LabelEncoder

for name , type_ in d_types.items():
    if str(type_) == 'object':
        Le = LabelEncoder()
        data[name] = Le.fit_transform(data[name])

Check info after Label Encoding

In [ ]:
data.info()

Check the feature correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

corr = data.corr()
sns.heatmap(corr , annot = True , ax=ax)

One hot Encoding 

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder()
part = onehotencoder.fit_transform(data['region'].values.reshape(-1,1)).toarray()

values = dict(data["region"].value_counts())

for e , (val , _) in enumerate(values.items()):
    data["region_" + str(val)] = part[:,e]

data = data.drop(["region"] , axis = 1)

data.head()

In [ ]:
data.info()

Handle Skewness in Predictive column

In [ ]:
Original_Y = data["expenses"].values.copy()

In [ ]:
Original_Y

In [ ]:
print("Skewness in Column : Expenses " , data["expenses"].skew())

plt.hist(data["expenses"])
plt.show()

In [ ]:
col_log = np.log(data["expenses"])
print("Skewness in Column : Log Expenses " , col_log.skew())

plt.hist(col_log)
plt.show()

In [ ]:
col_sqrt = np.sqrt(data["expenses"])

print("Skewness in Column : Sqrt Expenses " ,col_sqrt.skew())

plt.hist(col_sqrt)
plt.show()

In [ ]:
from scipy import stats 

col_cox , lam = stats.boxcox(data["expenses"])[0:2]
print("Skewness in Column : Sqrt Expenses " ,pd.Series(col_cox).skew())

plt.hist(col_cox)
plt.show()

In [ ]:
data["expenses"] = col_cox

Make Features and Targets

In [ ]:
remaining_columns = list(data.columns)
remaining_columns.remove("expenses")

In [ ]:
X = data[remaining_columns].values 
Y = data['expenses'].values

In [ ]:
from sklearn.preprocessing import StandardScaler

Scaler = StandardScaler()
X = Scaler.fit_transform(X)

In [ ]:
# check whether data is standardized or not 
# mean should be 1 

plt.ylim(-1,1)

means = []
for i in range(X.shape[1]):
    means.append(np.mean(X[:,i]))
plt.plot(means , scaley=False)

In [ ]:
# Check variances 

plt.ylim(0,2)

vars = []
for i in range(X.shape[1]):
    vars.append(np.var(X[:,i]))
plt.plot(vars)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA 

pca = PCA(n_components = 7)
X = pca.fit_transform(X)

pca.explained_variance_ratio_.cumsum()

In [ ]:
from sklearn.model_selection import KFold 

k_fold = KFold(n_splits=5)

test_scores = []
for train_idx , test_idx in k_fold.split(X):
    Xtrain = X[train_idx]
    Ytrain = Y[train_idx]

    Xtest = X[test_idx]
    Ytest = Y[test_idx]

    model = LinearRegression()
    model.fit(Xtrain , Ytrain)

    test_scores.append(model.score(Xtest , Ytest))

In [ ]:
print(" mean score of k folds : " , np.mean(test_scores))

plt.plot(test_scores)
plt.plot([np.mean(test_scores)]*len(test_scores))
plt.show()

Can we Bring back the data?

In [ ]:
from scipy.special import inv_boxcox

Real_data = inv_boxcox(Y , lam)

In [ ]:
Real_data[:10]

In [ ]:
Original_Y[:10]